In [9]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

# load + index
df = pd.read_csv('dataset/ec2_cpu_utilization_5f5533.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp').sort_index()

# labels
with open('dataset/combined_windows.json') as f:
    labels = json.load(f)
key = 'realAWSCloudwatch/ec2_cpu_utilization_5f5533.csv'
windows = [[pd.to_datetime(s), pd.to_datetime(e)] for s, e in labels[key]]

# ground-truth mask
mask = pd.Series(False, index=df.index)
for start, end in windows:
    mask |= (df.index >= start) & (df.index <= end)
df['is_anomaly'] = mask

In [10]:
df

,value,is_anomaly
timestamp,,
2014-02-14 14:27:00,51.846,False
2014-02-14 14:32:00,44.508,False
2014-02-14 14:37:00,41.244,False
2014-02-14 14:42:00,48.568,False
2014-02-14 14:47:00,46.714,False
...,...,...
2014-02-28 14:02:00,38.474,False
2014-02-28 14:07:00,40.352,False
2014-02-28 14:12:00,37.912,False


In [11]:
mask.mean()

np.float64(0.09970238095238096)

In [12]:
roll_mean = df['value'].rolling(12).mean()
df['roll_std'] = df['value'].rolling(12).std()
df['abs_dev'] = (df['value'] - roll_mean).abs()
df['change'] = df['value'].diff().abs()

# rolling features are NaN for the first 11 rows, drop them
df = df.dropna().copy()

In [13]:
df.shape

(4021, 5)

In [14]:
features = ['abs_dev', 'roll_std', 'change']
X = df[features]

model = IsolationForest(contamination=0.10, random_state=42)
df['predicted'] = model.fit_predict(X)

# sklearn returns -1 for anomaly, 1 for normal. flip to boolean.
df['predicted_anomaly'] = df['predicted'] == -1

print(f"Flagged {df['predicted_anomaly'].sum()} points as anomalies")

Flagged 402 points as anomalies


In [15]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

y_true = df['is_anomaly']
y_pred = df['predicted_anomaly']

print("Precision:", round(precision_score(y_true, y_pred), 3))
print("Recall:   ", round(recall_score(y_true, y_pred), 3))
print("F1:       ", round(f1_score(y_true, y_pred), 3))
print()
print(confusion_matrix(y_true, y_pred))

Precision: 0.172
Recall:    0.172
F1:        0.172

[[3286  333]
 [ 333   69]]
